# K-fold by ingredient and the all-data model (Colab)

Seven folds of the published ingredient hash (`scripts/14_kfold.py`; fold 0's test set is exactly the published test set). Each fold trains the recipe on five folds, picks its epoch on one, and is tested once on the seventh; pooling the seven test sets gives one out-of-fold accuracy over every VA string that is ever held out. Then the all-data model trains on every ingredient for the median best epoch of the folds and uploads its weights as the artifact `vandf-rxnorm-biencoder-final-all` (the published `vandf-rxnorm-biencoder` is never touched).

**Before running:** Runtime → **A100 GPU**. About two minutes per fold with the VANDF-only recipe, eight with MTHSPL. The folders were uploaded locally (`uv run scripts/14_kfold.py build [--sources VANDF,MTHSPL]` then `upload`). Set `RECIPE` in cell 5 to the winning arm of the levers sweep.

## 1. Check the GPU
Expect an A100. If this prints nothing, the runtime type is still CPU.

In [ ]:
!nvidia-smi -L

## 2. Get the code
A plain clone of the public repo. `BRANCH` lets this run from a feature branch before it is merged; re-running the cell pulls the latest commit.

In [ ]:
import os, subprocess

REPO = "kvenanzi/rxnorm"
BRANCH = "levers"
URL = f"https://github.com/{REPO}.git"
if not os.path.isdir("rxnorm"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, URL], check=True)
else:
    subprocess.run(["git", "-C", "rxnorm", "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", "rxnorm", "pull", "-q"], check=True)
!git -C rxnorm log --oneline -1

## 3. Make the package importable and install what Colab lacks
The clone directory goes on `sys.path`, so `import rxnorm_vandf` reads the code straight from the clone. The pip line adds only what Colab doesn't already ship, without upgrading what it does.

In [ ]:
import sys
if os.path.abspath("rxnorm") not in sys.path:
    sys.path.insert(0, os.path.abspath("rxnorm"))
%pip install -q duckdb sentence-transformers datasets wandb accelerate
import torch, sentence_transformers, numpy, pandas, rxnorm_vandf
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "| sentence-transformers", sentence_transformers.__version__,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__,
      "| rxnorm_vandf from", os.path.dirname(rxnorm_vandf.__file__))

## 4. Log in to Weights & Biases
The API key comes from Colab Secrets (key icon, `WANDB_API_KEY`, *Notebook access* on). `wandb.login()` reads the environment variable, so nothing is pasted or printed.

In [ ]:
import wandb
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
wandb.login()

## 5. Train the seven folds
One subprocess per fold, so a crash in one fold does not take the rest; each fold logs its full predictions (top-20 per string) as an artifact, which is what the out-of-fold analysis reads. Rerunning the cell after a disconnect re-runs every fold (the ledger lives in the ephemeral clone), so use `--only foldN` to finish a partial session.

In [ ]:
RECIPE = "--train-sources VANDF --aux none"        # or e.g. "--train-sources VANDF,MTHSPL --aux strength"
!cd rxnorm && python scripts/14_kfold.py train --from-artifact {RECIPE}

## 6. Train the all-data model
No validation split, no test split: the epoch count is the median best epoch of the seven folds (read from W&B), and the weights are logged as a model artifact from this runtime.

In [ ]:
!cd rxnorm && python scripts/14_kfold.py final --from-artifact {RECIPE}

## 7. Afterwards
Locally: `uv run scripts/14_kfold.py oof` (per-fold table, pooled out-of-fold accuracy with a Wilson interval, coverage) and `uv run scripts/14_kfold.py calibrate-oof` (temperature and Platt layer on the pooled out-of-fold predictions, written as `calibration.json`; the leave-one-fold-out and fold-to-fold threshold transfer).